In [1]:
import pandas as pd
import datetime
import importlib

import utils
import utils.dates as dateutils

from rich.console import Console

#Recarrega módulos locais (tive de fazer isto para evitar erros)
importlib.reload(utils.dates)

console = Console()

#Configuração de exibição
pd.set_option('display.float_format', '{:.0f}'.format)

console.log("[bold blue]Inicialização concluída[/]")

[15:39:28] Inicialização concluída                                                                 ]8;id=449960;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\2899422788.py\2899422788.py]8;;\:]8;id=605050;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\2899422788.py#18\18]8;;\

In [2]:
#Leitura dos arquivos
console.log(
    f"[bold blue]Mês atual: {dateutils.actual_month()} | Atualizando a partir de {dateutils.last_month()}[/]"
)

with console.status("[bold green]Lendo bases de dados...[/]"):

    #Bases novas (extraídas do SAP 4 HANA hoje)
    df_me80fn_raw = pd.read_csv(r'arquivos/me80fncru.csv')
    df_ksb1_raw = pd.read_csv(r'arquivos/ksb1cru.csv')

    #Bases históricas consolidadas
    df_me80fn = pd.read_csv(r'arquivos/ME80FN SEMESTRE 1 2026.csv')
    df_ksb1 = pd.read_csv(r'arquivos/KSB1 SEMESTRE 1 2026.csv')

console.log(
    f"[bold blue]Leitura concluída | "
    f"ME80FN CRU: {df_me80fn_raw.shape[0]} | "
    f"KSB1 CRU: {df_ksb1_raw.shape[0]} | "
    f"ME80FN Base: {df_me80fn.shape[0]} | "
    f"KSB1 Base: {df_ksb1.shape[0]}[/]"
)

           Mês atual: Março | Atualizando a partir de Fevereiro                                     ]8;id=483159;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1232431454.py\1232431454.py]8;;\:]8;id=8521;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1232431454.py#2\2]8;;\

           Leitura concluída | ME80FN CRU: 200 | KSB1 CRU: 200 | ME80FN Base: 3026 | KSB1 Base:    ]8;id=536048;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1232431454.py\1232431454.py]8;;\:]8;id=931399;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1232431454.py#16\16]8;;\
           2160                                                                                                    

In [3]:
#Tratamento dos dados "crus"
with console.status("[bold green]Tratando e consolidando dados...[/]"):

    #Remoção de duplicatas
    df_me80fn.drop_duplicates(inplace=True)
    df_ksb1.drop_duplicates(inplace=True)

    #Conversão de datas
    df_me80fn['Data do documento'] = pd.to_datetime(
        df_me80fn['Data do documento'], dayfirst=True, errors='coerce'
    )

    #Contagem antes da atualização
    outdated_month_amount = df_me80fn[
        df_me80fn['Data do documento'].dt.to_period('M') == pd.Timestamp('today').to_period('M')
    ].shape[0]

    # Criação de chave única
    df_me80fn_raw['Chave'] = (
        df_me80fn_raw['Documento de compras'].astype(str) +
        df_me80fn_raw['Item'].astype(str)
    ).astype('int64')

    df_ksb1_raw['Documento de compras'] = df_ksb1_raw['Documento de compras'].astype('Int64')
    df_ksb1_raw['Chave'] = (
        df_ksb1_raw['Documento de compras'].fillna(1).astype(str) +
        df_ksb1_raw['Item'].astype(str)
    ).astype('int64')

    #Aqui eu renomeei porque haviam duas colunas de mesmo nome em cada dataframe
    df_ksb1_raw.rename(columns={'Data do documento': 'Data Doc'}, inplace=True)

    #Atualização das bases
    df_me80fn = pd.concat([df_me80fn, df_me80fn_raw], ignore_index=True)
    df_ksb1 = pd.concat([df_ksb1, df_ksb1_raw], ignore_index=True)

    # Garantia de unicidade
    df_me80fn.drop_duplicates(inplace=True)
    df_ksb1.drop_duplicates(inplace=True)

console.log(
    f"[bold blue]Bases atualizadas | ME80FN: {df_me80fn.shape[0]} | KSB1: {df_ksb1.shape[0]}[/]"
)

C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\3111503244.py:9: UserWarning: Parsing dates in %Y-%m-%d 
format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_me80fn['Data do documento'] = pd.to_datetime(

           Bases atualizadas | ME80FN: 3226 | KSB1: 2160                                           ]8;id=940668;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\3111503244.py\3111503244.py]8;;\:]8;id=64408;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\3111503244.py#41\41]8;;\

In [4]:
#Salvamento dos dados
with console.status("[bold green]Salvando dados atualizados...[/]"):
    df_me80fn.to_csv(r"arquivos/ME80FN SEMESTRE 1 2026 consolidado.csv", index=False)
    df_ksb1.to_csv(r"arquivos/KSB1 SEMESTRE 1 2026 consolidado.csv", index=False)

console.log(f"[bold blue]Dados salvos em {datetime.datetime.now()}[/]")

           Dados salvos em 2026-03-26 15:39:28.349809                                               ]8;id=804920;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1220255667.py\1220255667.py]8;;\:]8;id=806821;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1220255667.py#6\6]8;;\

In [5]:
#Preparação dos dados para a análise
with console.status("[bold green]Preparando dados para análise...[/]"):

    df_me80fn['Valor líquido pedido'] = pd.to_numeric(
        df_me80fn['Valor líquido pedido'], errors="coerce"
    )

    df_ksb1 = df_ksb1[df_ksb1['Nº ref.estorno'].isnull()]

    df_ksb1['Data Doc'] = pd.to_datetime(df_ksb1['Data Doc'], dayfirst=True, errors='coerce')
    df_ksb1['Data de lançamento'] = pd.to_datetime(df_ksb1['Data de lançamento'], dayfirst=True, errors='coerce')

    df_me80fn['Data do documento'] = pd.to_datetime(
        df_me80fn['Data do documento'], dayfirst=True, errors='coerce'
    )

actual_month_amount = df_me80fn[
    df_me80fn['Data do documento'].dt.to_period('M') == pd.Timestamp('today').to_period('M')
].shape[0]

console.log(f"[bold blue]Antes: {outdated_month_amount} | Depois: {actual_month_amount}[/]")

C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1935239283.py:10: UserWarning: Parsing dates in %Y-%m-%d 
format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_ksb1['Data Doc'] = pd.to_datetime(df_ksb1['Data Doc'], dayfirst=True, errors='coerce')

           Antes: 478 | Depois: 508                                                                ]8;id=844350;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1935239283.py\1935239283.py]8;;\:]8;id=962351;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1935239283.py#21\21]8;;\

In [6]:
#Aplicação das regras de negócio 
with console.status("[bold green]Aplicando regras de negócio...[/]"):

    #Identifica se houve lançamento
    df_me80fn['Lançado_ME80FN'] = df_me80fn['Chave'].isin(df_ksb1['Chave'])

    # Merge com dados de lançamento
    df_me80fn = df_me80fn.merge(
        df_ksb1[['Chave', 'Data Doc', 'Data de lançamento']],
        on='Chave',
        how='left'
    ).drop_duplicates(subset=['Chave'])

    #Criação de colunas de tempo
    df_me80fn['AnoMes_real'] = df_me80fn['Data do documento'].dt.to_period('M').dt.to_timestamp()
    df_me80fn['AnoMes'] = df_me80fn['AnoMes_real'].dt.strftime('%b-%y')

    #Agrupamento principal
    df_me80fn_agrupado = (
        df_me80fn.sort_values('Data do documento', ascending=False)
        .groupby('Documento de compras', as_index=False)
        .agg({
            'Valor líquido pedido': 'sum',
            'Lançado_ME80FN': 'all',
            'Data do documento': 'first',
            'AnoMes_real': 'first',
            'AnoMes': 'first',
            'Data Doc': 'first',
            'Data de lançamento': 'first'
        })
    )

console.log(f"[bold blue]Pedidos únicos: {df_me80fn_agrupado.shape[0]}[/]")

           Pedidos únicos: 998                                                                     ]8;id=328705;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1125954418.py\1125954418.py]8;;\:]8;id=905250;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\1125954418.py#33\33]8;;\

In [7]:
#Aqui eu crio a tabela analítica
with console.status("[bold green]Gerando tabela analítica...[/]"):

    unique_anomes_real = sorted(df_me80fn_agrupado['AnoMes_real'].unique())
    df_pivot = pd.DataFrame(index=unique_anomes_real)

    df_pivot.index = df_pivot.index.strftime('%b-%y')
    df_pivot.index.name = 'Mês/Ano'

    df_pivot['Pedidos'] = df_me80fn_agrupado.groupby('AnoMes')['Documento de compras'].count()
    df_pivot['Pedidos (R$)'] = df_me80fn_agrupado.groupby('AnoMes')['Valor líquido pedido'].sum()

    df_me80fn_agrupado['Categorizado'] = False

    condition_col1 = (
        (df_me80fn_agrupado['Data Doc'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M')) &
        (df_me80fn_agrupado['Data de lançamento'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M'))
    )
    df_me80fn_agrupado.loc[condition_col1, 'Categorizado'] = True
    df_pivot['Pedidos c/NF e Miro (dentro do mês)'] = df_me80fn_agrupado[condition_col1].groupby('AnoMes').size()
    df_pivot['Pedidos c/NF e Miro (dentro do mês) (R$)'] = df_me80fn_agrupado[condition_col1].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_1'] = (df_pivot['Pedidos c/NF e Miro (dentro do mês)'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_col2 = (
        ~df_me80fn_agrupado['Categorizado'] &
        (
            (df_me80fn_agrupado['Data Doc'].dt.to_period('M') < df_me80fn_agrupado['Data do documento'].dt.to_period('M')) |
            (df_me80fn_agrupado['Data de lançamento'].dt.to_period('M') < df_me80fn_agrupado['Data do documento'].dt.to_period('M'))
        )
    )
    df_me80fn_agrupado.loc[condition_col2, 'Categorizado'] = True
    df_pivot['Pedidos c/NF de períodos anteriores'] = df_me80fn_agrupado[condition_col2].groupby('AnoMes').size()
    df_pivot['Pedidos c/NF de períodos anteriores (R$)'] = df_me80fn_agrupado[condition_col2].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_2'] = (df_pivot['Pedidos c/NF de períodos anteriores'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_col3 = (
        ~df_me80fn_agrupado['Categorizado'] &
        (df_me80fn_agrupado['Data Doc'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M')) &
        (df_me80fn_agrupado['Data de lançamento'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M') + 1)
    )
    df_me80fn_agrupado.loc[condition_col3, 'Categorizado'] = True
    df_pivot['Pedidos c/NF mês corrente + Lçto M+1'] = df_me80fn_agrupado[condition_col3].groupby('AnoMes').size()
    df_pivot['Pedidos c/NF mês corrente + Lçto M+1 (R$)'] = df_me80fn_agrupado[condition_col3].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_3'] = (df_pivot['Pedidos c/NF mês corrente + Lçto M+1'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_col4 = (
        ~df_me80fn_agrupado['Categorizado'] &
        (df_me80fn_agrupado['Data Doc'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M')) &
        (df_me80fn_agrupado['Data de lançamento'].dt.to_period('M') >= df_me80fn_agrupado['Data do documento'].dt.to_period('M') + 2)
    )
    df_me80fn_agrupado.loc[condition_col4, 'Categorizado'] = True
    df_pivot['Pedidos c/NF mês corrente + Lçto M+2 e posteriores'] = df_me80fn_agrupado[condition_col4].groupby('AnoMes').size()
    df_pivot['Pedidos c/NF mês corrente + Lçto M+2 e posteriores (R$)'] = df_me80fn_agrupado[condition_col4].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_4'] = (df_pivot['Pedidos c/NF mês corrente + Lçto M+2 e posteriores'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_col5 = (
        ~df_me80fn_agrupado['Categorizado'] &
        (df_me80fn_agrupado['Data Doc'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M') + 1) &
        (df_me80fn_agrupado['Data de lançamento'].dt.to_period('M') == df_me80fn_agrupado['Data do documento'].dt.to_period('M') + 1)
    )
    df_me80fn_agrupado.loc[condition_col5, 'Categorizado'] = True
    df_pivot['Pedidos c/NF de mês posterior + Lçto M+1'] = df_me80fn_agrupado[condition_col5].groupby('AnoMes').size()
    df_pivot['Pedidos c/NF de mês posterior + Lçto M+1 (R$)'] = df_me80fn_agrupado[condition_col5].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_5'] = (df_pivot['Pedidos c/NF de mês posterior + Lçto M+1'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_col6 = (
        ~df_me80fn_agrupado['Categorizado'] &
        (df_me80fn_agrupado['Data Doc'].dt.to_period('M') >= df_me80fn_agrupado['Data do documento'].dt.to_period('M') + 2)
    )
    df_me80fn_agrupado.loc[condition_col6, 'Categorizado'] = True
    df_pivot['Pedidos c/NF de meses posteriores _ Lçto M+2'] = df_me80fn_agrupado[condition_col6].groupby('AnoMes').size()
    df_pivot['Pedidos c/NF de meses posteriores _ Lçto M+2 (R$)'] = df_me80fn_agrupado[condition_col6].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_6'] = (df_pivot['Pedidos c/NF de meses posteriores _ Lçto M+2'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_col7 = (
        ~df_me80fn_agrupado['Categorizado'] &
        (df_me80fn_agrupado['Lançado_ME80FN'] == False)
    )
    df_me80fn_agrupado.loc[condition_col7, 'Categorizado'] = True
    df_pivot['Pedidos sem NF / MIRO PENDENTE'] = df_me80fn_agrupado[condition_col7].groupby('AnoMes').size()
    df_pivot['Pedidos sem NF / MIRO PENDENTE (R$)'] = df_me80fn_agrupado[condition_col7].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_7'] = (df_pivot['Pedidos sem NF / MIRO PENDENTE'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    condition_outros = ~df_me80fn_agrupado['Categorizado']
    df_pivot['Outros'] = df_me80fn_agrupado[condition_outros].groupby('AnoMes').size()
    df_pivot['Outros (R$)'] = df_me80fn_agrupado[condition_outros].groupby('AnoMes')['Valor líquido pedido'].sum()
    df_pivot['%_Outros'] = (df_pivot['Outros'] / df_pivot['Pedidos'] * 100).fillna(0)
    
    df_pivot = df_pivot.fillna(0)
    
    temp_index = pd.to_datetime(df_pivot.index, format='%b-%y')
    df_pivot = df_pivot.loc[temp_index.sort_values().strftime('%b-%y')]

    #Exportação final
    df_pivot = df_pivot.fillna(0)
    
    temp_index = pd.to_datetime(df_pivot.index, format='%b-%y')
    df_pivot = df_pivot.loc[temp_index.sort_values().strftime('%b-%y')]
    
    df_pivot.to_excel(r"arquivos/tabela_dinamica.xlsx")

console.log('[bold blue]Tabela dinâmica criada com sucesso! Linhas totais: '+str(df_pivot.shape[0])+'[/]')

Output()

           Tabela dinâmica criada com sucesso! Linhas totais: 6                                   ]8;id=53056;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\2381428237.py\2381428237.py]8;;\:]8;id=967226;file://C:\Users\murilo.silva\AppData\Local\Temp\ipykernel_21364\2381428237.py#102\102]8;;\